In [1]:
import numpy as np, pandas as pd
from scipy.spatial.distance import cdist

acc = "P01116"
res = pd.read_csv("../data/processed/P01116_residues.csv")
merged = pd.read_csv("../data/processed/P01116_merged.csv")

coords = {}
for line in open(f"../data/raw/{acc}_alphafold.pdb"):
    if line.startswith("ATOM") and line[12:16].strip() == "CA":
        coords[int(line[22:26])] = [float(line[30:38]), float(line[38:46]), float(line[46:54])]

# Compare within the folded part only (pLDDT >= 70), so the floppy tail doesn't distort things
pool = sorted(res.loc[res["plddt"] >= 70, "position"])
path_pos = sorted(set(merged.loc[merged["label"] == "pathogenic", "position"]) & set(pool))
print(len(path_pos), "unique pathogenic positions,", len(pool), "residues in the folded pool")

32 unique pathogenic positions, 175 residues in the folded pool


In [2]:
def mean_nn(positions):
    X = np.array([coords[p] for p in positions])
    d = cdist(X, X)
    np.fill_diagonal(d, np.inf)
    return d.min(axis=1).mean()      # average distance to the nearest other site

obs = mean_nn(path_pos)
rng = np.random.default_rng(0)
null = np.array([mean_nn(rng.choice(pool, len(path_pos), replace=False))
                 for _ in range(5000)])
pval = (np.sum(null <= obs) + 1) / (len(null) + 1)

print(f"Observed mean nearest-neighbor distance: {obs:.2f} Å")
print(f"Random expectation: {null.mean():.2f} ± {null.std():.2f} Å")
print(f"Permutation p-value (clustered): {pval:.4f}")

Observed mean nearest-neighbor distance: 5.21 Å
Random expectation: 5.88 ± 0.38 Å
Permutation p-value (clustered): 0.0370


In [3]:
counts = merged[merged["label"] == "pathogenic"].groupby("position").size()
print(counts.sort_values(ascending=False).head(10))

position
12     7
61     7
13     7
146    6
60     5
74     5
50     5
22     4
59     4
147    4
dtype: int64


In [4]:
from sklearn.cluster import DBSCAN

X = np.array([coords[p] for p in path_pos])
labels = DBSCAN(eps=8.0, min_samples=3).fit_predict(X)

out = pd.DataFrame({"position": path_pos, "cluster": labels})
print(out["cluster"].value_counts().sort_index())
for c in sorted(set(labels) - {-1}):
    print(f"Cluster {c}:", list(out.loc[out.cluster == c, "position"]))

cluster
-1     3
 0    29
Name: count, dtype: int64
Cluster 0: [5, 7, 12, 13, 14, 15, 18, 19, 22, 23, 34, 36, 58, 59, 60, 61, 63, 65, 68, 71, 72, 74, 77, 116, 117, 119, 146, 147, 153]


In [5]:
for eps in [4.5, 5.5, 6.5, 7.0]:
    labels = DBSCAN(eps=eps, min_samples=3).fit_predict(X)
    out = pd.DataFrame({"position": path_pos, "cluster": labels})
    n = len(set(labels) - {-1})
    print(f"\neps={eps}: {n} clusters, {(labels == -1).sum()} unclustered")
    for c in sorted(set(labels) - {-1}):
        print(f"  Cluster {c}:", list(out.loc[out.cluster == c, "position"]))
    print("  Unclustered:", list(out.loc[out.cluster == -1, "position"]))


eps=4.5: 1 clusters, 24 unclustered
  Cluster 0: [12, 13, 14, 15, 58, 59, 60, 61]
  Unclustered: [5, 7, 18, 19, 22, 23, 34, 36, 50, 63, 65, 68, 71, 72, 74, 77, 97, 116, 117, 119, 146, 147, 153, 164]

eps=5.5: 3 clusters, 11 unclustered
  Cluster 0: [12, 13, 14, 15, 18, 19, 22, 23, 36, 58, 59, 60, 61, 146, 147]
  Cluster 1: [71, 72, 74]
  Cluster 2: [116, 117, 119]
  Unclustered: [5, 7, 34, 50, 63, 65, 68, 77, 97, 153, 164]

eps=6.5: 2 clusters, 7 unclustered
  Cluster 0: [12, 13, 14, 15, 18, 19, 22, 23, 36, 58, 59, 60, 61, 63, 65, 68, 71, 72, 74, 146, 147, 153]
  Cluster 1: [116, 117, 119]
  Unclustered: [5, 7, 34, 50, 77, 97, 164]

eps=7.0: 2 clusters, 3 unclustered
  Cluster 0: [5, 7, 77]
  Cluster 1: [12, 13, 14, 15, 18, 19, 22, 23, 34, 36, 58, 59, 60, 61, 63, 65, 68, 71, 72, 74, 116, 117, 119, 146, 147, 153]
  Unclustered: [50, 97, 164]


In [6]:
labels = DBSCAN(eps=5.5, min_samples=3).fit_predict(X)
out = pd.DataFrame({"position": path_pos, "cluster": labels})
out.to_csv("../data/processed/P01116_clusters_eps5.5.csv", index=False)
print(out["cluster"].value_counts().sort_index())

cluster
-1    11
 0    15
 1     3
 2     3
Name: count, dtype: int64


In [ ]:
import sys
sys.path.append("..")
from pathlib import Path
from src.viz import show_clusters

clusters = pd.read_csv("../data/processed/P01116_clusters_eps5.5.csv")
view = show_clusters(acc, clusters)
view.show()
Path("../results/annotated_structures/KRAS_clusters.html").resolve().write_text(view._make_html())

In [ ]:
import json
data = json.load(open(f"../data/raw/{acc}_uniprot.json"))
for f in data["features"]:
    if f["type"] in ("Binding site", "Nucleotide binding", "Region", "Motif", "Domain", "Site"):
        s, e = f["location"]["start"]["value"], f["location"]["end"]["value"]
        print(f"{f['type']:<20} {s:>4}-{e:<4} {f.get('description', '')}")